In [1]:
import warnings
import os

warnings.filterwarnings("ignore")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [2]:
import random
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)
pd.set_option("display.precision", 4)

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f" Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA version: {torch.version.cuda}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    device = torch.device("cpu")
    print(" CUDA not available, using CPU")

print(f"Device: {device}")


 Using GPU: NVIDIA GeForce MX110
  CUDA version: 12.4
  VRAM: 2.00 GB
Device: cuda


In [3]:
dataset = load_dataset("dair-ai/emotion")

print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

labels = dataset['train'].features['label'].names
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
print(f"Labels ({len(labels)} classes): {labels}")

samples = dataset['train'].select(range(5))
print("\n--- Sample texts ---")
for i, sample in enumerate(samples):
    print(f"{i+1}. Text: {sample['text'][:100]}...")
    print(f"   Label: {labels[sample['label']]}")


Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 90139.02 examples/s]


Train size: 16000
Validation size: 2000
Test size: 2000
Labels (6 classes): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

--- Sample texts ---
1. Text: i didnt feel humiliated...
   Label: sadness
2. Text: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and ...
   Label: sadness
3. Text: im grabbing a minute to post i feel greedy wrong...
   Label: anger
4. Text: i am ever feeling nostalgic about the fireplace i will know that it is still on the property...
   Label: love
5. Text: i am feeling grouchy...
   Label: anger


In [4]:
MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Model max length: {tokenizer.model_max_length}")

print("\n--- Tokenization Examples ---")
for i in range(3):
    text = samples[i]['text']
    print(f"\nExample {i+1}: {text[:80]}...")
    
    encoded = tokenizer(text, add_special_tokens=True, return_attention_mask=True, return_token_type_ids=True)
    full_ids = encoded["input_ids"]
    full_tokens = tokenizer.convert_ids_to_tokens(full_ids)
    
    df_tokens = pd.DataFrame({
        "position": list(range(len(full_ids))),
        "token": full_tokens,
        "token_id": full_ids,
        "attention_mask": encoded["attention_mask"],
        "token_type_id": encoded.get("token_type_ids", [0]*len(full_ids)),
    })
    display(df_tokens.head(15))
    
    special_info = []
    if full_tokens[0] == tokenizer.cls_token:
        special_info.append(f"[CLS] at start (id={tokenizer.cls_token_id})")
    if full_tokens[-1] == tokenizer.sep_token:
        special_info.append(f"[SEP] at end (id={tokenizer.sep_token_id})")
    if special_info:
        print(f"Special tokens: {', '.join(special_info)}")

Tokenizer loaded: cointegrated/rubert-tiny2
Model max length: 2048

--- Tokenization Examples ---

Example 1: i didnt feel humiliated...


,position,token,token_id,attention_mask,token_type_id
0,0,[CLS],2,1,0
1,1,i,76,1,0
2,2,didn,11055,1,0
3,3,##t,549,1,0
4,4,feel,12235,1,0
5,5,hu,8338,1,0
6,6,##mil,17141,1,0
7,7,##iated,25292,1,0
8,8,[SEP],3,1,0


Special tokens: [CLS] at start (id=2), [SEP] at end (id=3)

Example 2: i can go from feeling so hopeless to so damned hopeful just from being around so...


,position,token,token_id,attention_mask,token_type_id
0,0,[CLS],2,1,0
1,1,i,76,1,0
2,2,can,1147,1,0
3,3,go,1695,1,0
4,4,from,610,1,0
5,5,feeling,18804,1,0
6,6,so,773,1,0
7,7,hope,15939,1,0
8,8,##less,3425,1,0
9,9,to,540,1,0


Special tokens: [CLS] at start (id=2), [SEP] at end (id=3)

Example 3: im grabbing a minute to post i feel greedy wrong...


,position,token,token_id,attention_mask,token_type_id
0,0,[CLS],2,1,0
1,1,im,631,1,0
2,2,gra,19420,1,0
3,3,##bbi,12169,1,0
4,4,##ng,770,1,0
5,5,a,68,1,0
6,6,minute,6494,1,0
7,7,to,540,1,0
8,8,post,1729,1,0
9,9,i,76,1,0


Special tokens: [CLS] at start (id=2), [SEP] at end (id=3)


In [5]:
PRETRAINED_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
pretrained_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
pretrained_model = AutoModelForSequenceClassification.from_pretrained(PRETRAINED_MODEL).to(device)
pretrained_model.eval()

print(f"\nPretrained model: {PRETRAINED_MODEL}")
print(f"Labels: {pretrained_model.config.id2label}")

def predict_text_manual(model, tokenizer, text: str, device, id2label) -> Dict[str, Any]:
    encoded = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**encoded)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        pred_id = int(torch.argmax(probs, dim=-1).item())
        pred_label = id2label[pred_id]
        confidence = float(probs[0, pred_id].item())
    return {"text": text, "pred_label": pred_label, "confidence": confidence}

print("\n--- Inference Comparison (Pipeline vs Manual) ---")
test_texts = [samples[i]['text'] for i in range(5)]

clf_pipeline = pipeline("text-classification", model=pretrained_model, tokenizer=pretrained_tokenizer, device=0 if torch.cuda.is_available() else -1)
pipeline_results = clf_pipeline(test_texts)
manual_results = [predict_text_manual(pretrained_model, pretrained_tokenizer, t, device, pretrained_model.config.id2label) for t in test_texts]

comparison_df = pd.DataFrame({
    "text": [t[:60] + "..." for t in test_texts],
    "pipeline_label": [r["label"] for r in pipeline_results],
    "pipeline_score": [f"{r['score']:.3f}" for r in pipeline_results],
    "manual_label": [r["pred_label"] for r in manual_results],
    "manual_confidence": [f"{r['confidence']:.3f}" for r in manual_results],
    "match": [pipeline_results[i]["label"] == manual_results[i]["pred_label"] for i in range(5)]
})
display(comparison_df)

print("\n--- Note ---")
print(f"Pretrained labels: {list(pretrained_model.config.id2label.values())}")
print(f"Our task labels: {labels}")
print("=> Mismatch confirms need for fine-tuning.")


Pretrained model: nlptown/bert-base-multilingual-uncased-sentiment
Labels: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}

--- Inference Comparison (Pipeline vs Manual) ---


,text,pipeline_label,pipeline_score,manual_label,manual_confidence,match
0,i didnt feel humiliated...,2 stars,0.400,2 stars,0.400,True
1,i can go from feeling so hopeless to so damned hopeful just ...,1 star,0.464,1 star,0.464,True
2,im grabbing a minute to post i feel greedy wrong...,1 star,0.605,1 star,0.605,True
3,i am ever feeling nostalgic about the fireplace i will know ...,5 stars,0.537,5 stars,0.537,True
4,i am feeling grouchy...,2 stars,0.383,2 stars,0.383,True



--- Note ---
Pretrained labels: ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']
Our task labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
=> Mismatch confirms need for fine-tuning.


In [6]:
def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    return tokenizer(batch["text"], truncation=True, max_length=128)

hf_dataset = DatasetDict({
    "train": dataset["train"],
    "validation": dataset["validation"],
    "test": dataset["test"],
})
tokenized_datasets = hf_dataset.map(tokenize_batch, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print(f"Tokenized datasets: {tokenized_datasets}")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(device)
print(f"Model loaded: {MODEL_NAME}, labels: {model.config.num_labels}")

Map: 100%|██████████| 2000/2000 [00:00<00:00, 3325.23 examples/s]


Tokenized datasets: DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})


BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

Model loaded: cointegrated/rubert-tiny2, labels: 6


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

common_kwargs = dict(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,      
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),     
)

try:
    training_args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common_kwargs)
except TypeError:
    training_args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common_kwargs)

try:
    trainer = Trainer(
        model=model, args=training_args, train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"], processing_class=tokenizer,
        data_collator=data_collator, compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model, args=training_args, train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"], tokenizer=tokenizer,
        data_collator=data_collator, compute_metrics=compute_metrics,
    )

print("\n--- Starting Fine-tuning ---")
print(f"Training on {device} with fp16={common_kwargs['fp16']}")
train_result = trainer.train()
print(f"Training completed. Train loss: {train_result.metrics.get('train_loss', 'N/A')}")


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [ ]:
import os
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

print("\n--- Evaluation on Test Set ---")
test_metrics = trainer.evaluate(tokenized_datasets["test"])
print(f"Test Accuracy: {test_metrics['eval_accuracy']:.4f}")
print(f"Test F1 Macro: {test_metrics['eval_f1_macro']:.4f}")

test_output = trainer.predict(tokenized_datasets["test"])
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_true = test_output.label_ids
test_probs = F.softmax(torch.tensor(test_logits), dim=-1).numpy()

notebook_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
artifacts_dir = os.path.join(notebook_dir, "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)  

sample_indices = random.sample(range(len(test_preds)), 10)
pred_rows = []
for idx in sample_indices:
    pred_rows.append({
        "text": dataset["test"][idx]["text"],
        "true_label": labels[test_true[idx]],
        "pred_label": labels[test_preds[idx]],
        "confidence": float(test_probs[idx][test_preds[idx]]),
    })
df_predictions = pd.DataFrame(pred_rows)
df_predictions.to_csv(os.path.join(artifacts_dir, "sample_predictions.csv"), index=False)
print(f" Saved {artifacts_dir}/sample_predictions.csv")

cm = confusion_matrix(test_true, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(artifacts_dir, "confusion_matrix.png"), dpi=150)
plt.close()
print(f" Saved {artifacts_dir}/confusion_matrix.png")

In [ ]:
print("\n--- Error Analysis Examples ---")
error_indices = [i for i in range(len(test_preds)) if test_preds[i] != test_true[i]]
if error_indices:
    for idx in error_indices[:5]:
        text = dataset["test"][idx]["text"]
        true_lbl = labels[test_true[idx]]
        pred_lbl = labels[test_preds[idx]]
        conf = test_probs[idx][test_preds[idx]]
        print(f"Text: {text[:100]}...")
        print(f"True: {true_lbl} | Pred: {pred_lbl} (confidence: {conf:.3f})")
        print("-" * 80)
else:
    print("No errors found.")

print("\n=== FINAL METRICS FOR REPORT ===")
print(f"Accuracy: {test_metrics['eval_accuracy']:.4f}")
print(f"F1 Macro: {test_metrics['eval_f1_macro']:.4f}")
